# Exercise 5 — Full Productization Pipeline

Apply all four components to the Section 7 trading bot: generate a pyproject.toml, a README, a changelog entry, pricing tiers, and verify the feature gate. This is what 'productizing' looks like in practice — generating all the artifacts that make a script into a distributable product.

In [ ]:
from dataclasses import dataclass, field
import datetime

@dataclass
class ProductConfig:
    name: str; version: str; description: str
    author: str; email: str
    dependencies: list = field(default_factory=list)
    license: str  = "MIT"
    python_requires: str = ">=3.10"

_CFG = ProductConfig(
    name="my-trading-bot",
    version="0.1.0",
    description="AI-powered paper-trading bot using sentiment and technical signals.",
    author="Jane Doe",
    email="jane@example.com",
    dependencies=["pandas>=2.0", "requests>=2.28"],
)
def generate_pyproject(cfg):
    cli_name = cfg.name.replace("-", "_")
    if cfg.dependencies:
        deps_lines = "\n".join(f'    "{d}",' for d in cfg.dependencies)
        deps_block = f"[\n{deps_lines}\n]"
    else:
        deps_block = "[]"
    return (
        f"[build-system]\n"
        f'requires = ["setuptools>=68", "wheel"]\n'
        f'build-backend = "setuptools.backends.legacy:build"\n'
        f"\n[project]\n"
        f'name = "{cfg.name}"\n'
        f'version = "{cfg.version}"\n'
        f'description = "{cfg.description}"\n'
        f'authors = [{{name = "{cfg.author}", email = "{cfg.email}"}}]\n'
        f'license = {{text = "{cfg.license}"}}\n'
        f'requires-python = "{cfg.python_requires}"\n'
        f"dependencies = {deps_block}\n"
        f"\n[project.scripts]\n"
        f'{cli_name} = "{cli_name}:main"\n'
    )
def generate_readme(cfg, features, examples):
    feature_bullets = "\n".join(f"- {f}" for f in features)
    usage_blocks    = "\n\n".join(
        f"### {ex['title']}\n```python\n{ex['code']}\n```"
        for ex in examples
    )
    return (
        f"# {cfg.name}\n\n{cfg.description}\n\n"
        f"## Features\n\n{feature_bullets}\n\n"
        f"## Installation\n\npip install {cfg.name}\n\n"
        f"## Usage\n\n{usage_blocks}\n\n"
        f"## License\n\n{cfg.license}\n"
    )

def generate_changelog_entry(version, changes):
    today = datetime.date.today().isoformat()
    change_list = "\n".join(f"- {c}" for c in changes)
    return f"## [{version}] — {today}\n\n{change_list}\n"
def calculate_price(cost_per_call, calls_per_month, margin=0.5):
    if margin >= 1.0 or margin < 0:
        raise ValueError(f"margin must be in [0,1); got {margin}")
    return round(cost_per_call * calls_per_month / (1.0 - margin), 2)

def generate_pricing_tiers(cost_per_call, tiers):
    result = []
    for tier in tiers:
        calls  = tier["calls"]; margin = tier.get("margin", 0.5)
        price  = calculate_price(cost_per_call, calls, margin)
        result.append({
            "name": tier["name"], "calls": calls,
            "price_per_month": price,
            "price_per_call":  round(price / max(calls, 1), 6),
        })
    return result

def format_pricing_table(tiers):
    header = "| Plan | Calls/month | Price/month | Price/call |"
    sep    = "|------|-------------|-------------|------------|"
    rows   = [
        f"| {t['name']} | {t['calls']:,} | ${t['price_per_month']:.2f} | "
        f"${t['price_per_call']:.6f} |"
        for t in tiers
    ]
    return "\n".join([header, sep] + rows)
def validate_api_key(key, valid_keys):
    return key in valid_keys

def gate_feature(api_key, valid_keys, feature_fn, *args, **kwargs):
    if not validate_api_key(api_key, valid_keys):
        raise PermissionError(f"Invalid API key: {api_key!r}")
    return feature_fn(*args, **kwargs)


In [ ]:
# Trading bot product config
bot_cfg = ProductConfig(
    name        = "ai-trading-bot",
    version     = "1.0.0",
    description = "AI-powered paper-trading bot with sentiment and technical signals.",
    author      = "AI Engineer",
    email       = "bot@example.com",
    dependencies= ["pandas>=2.0", "requests>=2.28"],
)
features = [
    "SMA crossover and RSI mean-reversion strategies",
    "AI sentiment analysis from news headlines",
    "Kelly Criterion position sizing",
    "Stop-loss and drawdown limit risk controls",
    "Paper-trading bot with daily scheduling and logging",
]
examples = [
    {"title": "Quick Start",
     "code": "from ai_trading_bot import BotRunner\nrunner = BotRunner('bot.log')"},
    {"title": "Run paper trader",
     "code": "result = runner.run_once(df, signals, initial_cash=10_000)"},
]
tiers = [
    {"name": "Free",       "calls":    100, "margin": 0.50},
    {"name": "Hobbyist",   "calls":  1_000, "margin": 0.55},
    {"name": "Pro",        "calls": 10_000, "margin": 0.65},
    {"name": "Enterprise", "calls": 50_000, "margin": 0.70},
]
VALID_KEYS = {"demo-key-001", "demo-key-002"}

# Generate artifacts
pyproject  = generate_pyproject(bot_cfg)
readme     = generate_readme(bot_cfg, features, examples)
changelog  = generate_changelog_entry("1.0.0", ["Initial release"])
tier_data  = generate_pricing_tiers(0.001, tiers)
price_table = format_pricing_table(tier_data)

print("=== pyproject.toml ===")
print(pyproject)
print("=== Pricing Table ===")
print(price_table)


### Checks

In [ ]:
checks = 0

# 1 — pyproject has correct name and version
try:
    assert "ai-trading-bot" in pyproject and "1.0.0" in pyproject
    assert "[project]" in pyproject
    checks += 1; print("✅ 1 pyproject contains name, version, [project]")
except Exception as e:
    print("❌ 1:", e)

# 2 — readme has all major sections
try:
    for section in ["## Features", "## Installation", "## Usage", "## License"]:
        assert section in readme, f"missing {section}"
    assert "ai-trading-bot" in readme
    checks += 1; print("✅ 2 README has all required sections")
except Exception as e:
    print("❌ 2:", e)

# 3 — pricing tiers: 4 tiers, prices increase with volume
try:
    assert len(tier_data) == 4
    prices = [t["price_per_month"] for t in tier_data]
    assert all(prices[i] < prices[i+1] for i in range(len(prices)-1)),         f"prices should increase with volume: {prices}"
    checks += 1; print(f"✅ 3 4 tiers, prices increase: {[f'${p:.2f}' for p in prices]}")
except Exception as e:
    print("❌ 3:", e)

# 4 — pricing table is a markdown table
try:
    lines = price_table.split("\n")
    assert lines[0].startswith("|") and lines[0].endswith("|")
    assert "---" in lines[1]
    assert len(lines) >= 6  # header + sep + 4 tiers
    checks += 1; print("✅ 4 format_pricing_table returns a valid Markdown table")
except Exception as e:
    print("❌ 4:", e)

# 5 — feature gate
try:
    result = gate_feature("demo-key-001", VALID_KEYS, lambda: "premium data")
    assert result == "premium data"
    try:
        gate_feature("bad-key", VALID_KEYS, lambda: "premium data")
        print("❌ 5: expected PermissionError for bad key")
    except PermissionError:
        checks += 1; print("✅ 5 gate_feature: valid key → result; invalid → PermissionError")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
